In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/DataEngineering-Project/de_project/src/includes"

In [0]:
# dbutils.widgets.text("catalog","de_dev")

In [0]:
catalog = dbutils.widgets.get("catalog")
print(catalog)

In [0]:
df = spark.table(f"{catalog}.bronze.products")
clean_df = (
    df
    .dropna(subset=["product_id" , "product_name","price","supplier_id"])
    .fillna({"category":"Unknown"})
    .dropDuplicates(["product_id" , "product_name"])
    .withColumn("product_name", trim(col("product_name")))
    .select(
        "product_id" , "product_name","category","price","supplier_id","sku"
    )
    )
clean_df.createOrReplaceTempView("products_clean_view")
# display(clean_df)


In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.silver.products_scd_2(
    product_id int,
    product_name string,
    category string,
    price decimal(10,2),
    supplier_id int,
    sku string,

    effective_date date,
    end_date date,
    is_current boolean,
    version int
)
USING DELTA;

In [0]:
%sql
MERGE INTO ${catalog}.silver.products_scd_2 AS t
USING (
    SELECT
        product_id,
        product_name,
        category,
        price,
        supplier_id
    FROM products_clean_view
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY product_id
        ORDER BY product_id
    ) = 1
) AS s
ON t.product_id = s.product_id
AND t.is_current = true

WHEN MATCHED
AND (
    t.product_name != s.product_name OR
    t.category != s.category OR
    t.price != s.price OR
    t.supplier_id != s.supplier_id
)
THEN UPDATE SET
    t.end_date = current_date(),
    t.is_current = false;

In [0]:
%sql
INSERT INTO ${catalog}.silver.products_scd_2
SELECT
    s.product_id,
    s.product_name,
    s.category,
    s.price,
    s.supplier_id,
    s.sku,
    current_date() AS effective_date,
    NULL AS end_date,
    TRUE AS is_current,
    COALESCE(t.version, 0) + 1 AS version
FROM (
    SELECT
        product_id,
        product_name,
        category,
        price,
        supplier_id,
        sku
    FROM products_clean_view
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY product_id
        ORDER BY product_id
    ) = 1
) s

LEFT JOIN (
    SELECT
        product_id,
        MAX(version) AS version
    FROM ${catalog}.silver.products_scd_2
    GROUP BY product_id
) t
ON s.product_id = t.product_id

LEFT JOIN ${catalog}.silver.products_scd_2 c
ON s.product_id = c.product_id
AND c.is_current = true

WHERE
    c.product_id IS NULL
    OR
    c.product_name != s.product_name
    OR
    c.category != s.category
    OR
    c.price != s.price
    OR 
    c.supplier_id != s.supplier_id;